Team coursework: Logistic Regression classification on an anonymised Kaggle-style tabular dataset (class-imbalanced binary target), part of the same assignment as `svm_imbalance_and_tuning.ipynb` in this folder. The input CSV and rendered outputs are not included, and results are not independently reproduced in this portfolio copy.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

In [ ]:
df = pd.read_csv("data/datafile_small.csv")
df.head()

# Retrial (1/12)

In [ ]:
# ==========================================
# STEP 1: DROP ID_CODE
# ==========================================

print("\n" + "-"*80)
print("STEP 1: DROP ID_CODE")
print("-"*80)

if 'ID_code' in df.columns:
    df = df.drop(columns=['ID_code'])
    print("✅ Dropped ID_code column")
    print(f"   Reason: Identifier column, not a predictive feature")
else:
    print("⚠️  ID_code column not found (already dropped)")

print(f"\nCurrent shape: {df.shape}")

# ==========================================
# STEP 2: DROP VAR_45 (HIGH MISSING VALUES)
# ==========================================

print("\n" + "-"*80)
print("STEP 2: DROP VAR_45 (HIGH MISSING VALUES)")
print("-"*80)

if 'var_45' in df.columns:
    missing_count = df['var_45'].isnull().sum()
    missing_pct = (missing_count / len(df)) * 100
    print(f"\nvar_45 statistics:")
    print(f"   Missing values: {missing_count:,}")
    print(f"   Missing percentage: {missing_pct:.2f}%")

    df = df.drop(columns=['var_45'])
    print(f"\n✅ Dropped var_45 column")
    print(f"   Reason: {missing_pct:.1f}% missing (>40% threshold)")
else:
    print("⚠️  var_45 column not found (already dropped)")

print(f"\nCurrent shape: {df.shape}")

In [ ]:
# ==========================================
# STEP 3: VERIFY REMAINING FEATURES
# ==========================================

print("\n" + "-"*80)
print("STEP 3: VERIFY REMAINING FEATURES")
print("-"*80)

# Get feature columns (after dropping ID and var_45)
feature_cols = [col for col in df.columns if col.startswith('var_')]

print(f"\nFeature columns remaining: {len(feature_cols)}")
print(f"Target column: {'target' in df.columns}")

# Check remaining missing values
remaining_missing = df[feature_cols].isnull().sum().sum()
print(f"\nRemaining missing values in features: {remaining_missing:,}")

if remaining_missing > 0:
    missing_cols = df[feature_cols].columns[df[feature_cols].isnull().any()].tolist()
    print(f"Columns with missing values: {len(missing_cols)}")
    print(f"\nSummary (top 10):")
    for col in missing_cols[:10]:
        miss_count = df[col].isnull().sum()
        miss_pct = (miss_count / len(df)) * 100
        print(f"   {col}: {miss_count:,} ({miss_pct:.2f}%)")
    if len(missing_cols) > 10:
        print(f"   ... and {len(missing_cols) - 10} more")

## 1. Pick Sampling Method -> Class Weight

In [ ]:
X = df.drop(columns=['target'])
y = df['target']

randomseed = 30
TEST_SIZE = 0.2

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=randomseed, stratify=y)

# Impute missing values using SimpleImputer with the mean strategy
imputer = SimpleImputer(strategy='mean')
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

# Convert back to DataFrame (optional, but good for consistency)
X_train_imputed = pd.DataFrame(X_train_imputed, columns=X_train.columns)
X_test_imputed = pd.DataFrame(X_test_imputed, columns=X_test.columns)

print("Data split and imputed successfully!")
print(f"X_train_imputed shape: {X_train_imputed.shape}")
print(f"X_test_imputed shape: {X_test_imputed.shape}")

In [ ]:

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from imblearn.over_sampling import SMOTE
from imblearn.ensemble import EasyEnsembleClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, fbeta_score, matthews_corrcoef
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

print(f"\nTrain set: {X_train_scaled.shape[0]:,} samples")
print(f"Test set:  {X_test_scaled.shape[0]:,} samples")
print(f"\nClass distribution in training set:")
print(f"  Class 0: {(y_train==0).sum():,} ({(y_train==0).sum()/len(y_train)*100:.2f}%)")
print(f"  Class 1: {(y_train==1).sum():,} ({(y_train==1).sum()/len(y_train)*100:.2f}%)")

def calculate_all_metrics(y_true, y_pred, y_proba):
    """Calculate all metrics including PR-AUC, F2-Score, MCC"""
    return {
        'PR-AUC': average_precision_score(y_true, y_proba),
        'F2-Score': fbeta_score(y_true, y_pred, beta=2),
        'MCC': matthews_corrcoef(y_true, y_pred),
        'ROC-AUC': roc_auc_score(y_true, y_proba),
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred),
        'F1-Score': f1_score(y_true, y_pred)
    }

def print_full_results(y_true, y_pred, y_proba, method_name):
    metrics = calculate_all_metrics(y_true, y_pred, y_proba)

    print(f"\n{'='*60}")
    print(f"RESULTS: {method_name}")
    print(f"{'='*60}")
    print(f"\n  [Primary Metrics]")
    print(f"  PR-AUC:     {metrics['PR-AUC']:.4f}")
    print(f"  F2-Score:   {metrics['F2-Score']:.4f}")
    print(f"  MCC:        {metrics['MCC']:.4f}")
    print(f"\n  [Additional Metrics]")
    print(f"  ROC-AUC:    {metrics['ROC-AUC']:.4f}")
    print(f"  Accuracy:   {metrics['Accuracy']:.4f}")
    print(f"  Precision:  {metrics['Precision']:.4f}")
    print(f"  Recall:     {metrics['Recall']:.4f}")
    print(f"  F1-Score:   {metrics['F1-Score']:.4f}")

    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    print(f"\n  [Confusion Matrix]")
    print(f"                    Predicted")
    print(f"                  Neg      Pos")
    print(f"  Actual Neg    {cm[0,0]:5d}    {cm[0,1]:5d}")
    print(f"  Actual Pos    {cm[1,0]:5d}    {cm[1,1]:5d}")
    print(f"\n  TN: {cm[0,0]:,}  |  FP: {cm[0,1]:,}")
    print(f"  FN: {cm[1,0]:,}  |  TP: {cm[1,1]:,}")

    # Classification Report
    print(f"\n  [Classification Report]")
    report = classification_report(y_true, y_pred, digits=4)
    for line in report.split('\n'):
        print(f"  {line}")

    return metrics

# PREPARE SAMPLING METHODS
print("\n" + "="*80)
print("PREPARING SAMPLING METHODS")
print("="*80)

# Store results
step1_results = []

# 1. No Balancing (Baseline)
print("\n[1] No Balancing: Using original data")

# 2. SMOTE
smote = SMOTE(random_state=randomseed)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)
print(f"\n[2] SMOTE applied:")
print(f"    Before: {len(y_train):,} samples")
print(f"    After:  {len(y_train_smote):,} samples")

# 3. Class Weight
print("\n[3] Class Weight: 'balanced'")

# 4. EasyEnsemble
print("\n[4] EasyEnsemble: Uses balanced subsets with ensemble learning")

# MODEL TRAINING AND EVALUATION
print("\n" + "="*80)
print("MODEL TRAINING AND EVALUATION")
print("="*80)

# 1. NO BALANCING (BASELINE)
print("\n" + "#"*80)
print("# METHOD 1: NO BALANCING (BASELINE)")
print("#"*80)

model_1 = LogisticRegression(max_iter=1000, random_state=randomseed)
model_1.fit(X_train_scaled, y_train)

y_pred_1 = model_1.predict(X_test_scaled)
y_proba_1 = model_1.predict_proba(X_test_scaled)[:, 1]

metrics_1 = print_full_results(y_test, y_pred_1, y_proba_1, "No Balancing (Baseline)")
step1_results.append({
    'Method': 'No Balancing',
    **metrics_1
})

# 2. SMOTE
print("\n" + "#"*80)
print("# METHOD 2: SMOTE")
print("#"*80)

model_2 = LogisticRegression(max_iter=1000, random_state=randomseed)
model_2.fit(X_train_smote, y_train_smote)

y_pred_2 = model_2.predict(X_test_scaled)
y_proba_2 = model_2.predict_proba(X_test_scaled)[:, 1]

metrics_2 = print_full_results(y_test, y_pred_2, y_proba_2, "SMOTE")
step1_results.append({
    'Method': 'SMOTE',
    **metrics_2
})

# 3. CLASS WEIGHT
print("\n" + "#"*80)
print("# METHOD 3: CLASS WEIGHT (BALANCED)")
print("#"*80)

model_3 = LogisticRegression(max_iter=1000, random_state=randomseed, class_weight='balanced')
model_3.fit(X_train_scaled, y_train)

y_pred_3 = model_3.predict(X_test_scaled)
y_proba_3 = model_3.predict_proba(X_test_scaled)[:, 1]

metrics_3 = print_full_results(y_test, y_pred_3, y_proba_3, "Class Weight (Balanced)")
step1_results.append({
    'Method': 'Class Weight',
    **metrics_3
})

# 4. EASYENSEMBLE
print("\n" + "#"*80)
print("# METHOD 4: EASYENSEMBLE")
print("#"*80)

model_4 = EasyEnsembleClassifier(
    estimator=LogisticRegression(max_iter=1000, random_state=randomseed),
    n_estimators=10,
    random_state=randomseed,
    n_jobs=-1
)
model_4.fit(X_train_scaled, y_train)

y_pred_4 = model_4.predict(X_test_scaled)
y_proba_4 = model_4.predict_proba(X_test_scaled)[:, 1]

metrics_4 = print_full_results(y_test, y_pred_4, y_proba_4, "EasyEnsemble")
step1_results.append({
    'Method': 'EasyEnsemble',
    **metrics_4
})

# COMPARISON SUMMARY
print("\n" + "="*80)
print("STEP 1: COMPARISON SUMMARY")
print("="*80)

# Create comparison DataFrame
comparison_df = pd.DataFrame(step1_results)
comparison_df = comparison_df.set_index('Method')

# Reorder columns: Primary metrics first
column_order = ['PR-AUC', 'F2-Score', 'MCC', 'ROC-AUC', 'Accuracy', 'Precision', 'Recall', 'F1-Score']
comparison_df = comparison_df[column_order]
comparison_df = comparison_df.round(4)

# Sort by PR-AUC (primary metric)
comparison_df_sorted = comparison_df.sort_values('PR-AUC', ascending=False)

print("\n All Methods Comparison (Sorted by PR-AUC):\n")
print(comparison_df_sorted.to_string())

# RANKING BY EACH PRIMARY METRIC
print("\n" + "-"*80)
print("RANKING BY PRIMARY METRICS")
print("-"*80)

# Rank by PR-AUC
print("\n Ranking by PR-AUC:")
rank_prauc = comparison_df.sort_values('PR-AUC', ascending=False)[['PR-AUC']]
rank_prauc['Rank'] = range(1, len(rank_prauc)+1)
print(rank_prauc.to_string())

# Rank by F2-Score
print("\n Ranking by F2-Score:")
rank_f2 = comparison_df.sort_values('F2-Score', ascending=False)[['F2-Score']]
rank_f2['Rank'] = range(1, len(rank_f2)+1)
print(rank_f2.to_string())

# Rank by MCC
print("\n Ranking by MCC:")
rank_mcc = comparison_df.sort_values('MCC', ascending=False)[['MCC']]
rank_mcc['Rank'] = range(1, len(rank_mcc)+1)
print(rank_mcc.to_string())

In [ ]:
# BEST MODEL SELECTION (USER OVERRIDE)
print("\n" + "="*80)
print("🏆 BEST SAMPLING METHOD SELECTION (User Override)")
print("="*80)

# User specified best method
best_method = 'Class Weight'
best_metrics = comparison_df.loc[best_method]

print(f"\n  Best Method: {best_method}")
print(f"\n  Primary Metrics:")
print(f"    PR-AUC:    {best_metrics['PR-AUC']:.4f}")
print(f"    F2-Score:  {best_metrics['F2-Score']:.4f}")
print(f"    MCC:       {best_metrics['MCC']:.4f}")
print(f"\n  Additional Metrics:")
print(f"    ROC-AUC:   {best_metrics['ROC-AUC']:.4f}")
print(f"    Accuracy:  {best_metrics['Accuracy']:.4f}")
print(f"    Precision: {best_metrics['Precision']:.4f}")
print(f"    Recall:    {best_metrics['Recall']:.4f}")
print(f"    F1-Score:  {best_metrics['F1-Score']:.4f}")

# FINAL OUTPUT
print("\n" + "="*80)
print("STEP 1 COMPLETE")
print("="*80)
print(f"\n🏆 Best Sampling Method: {best_method}")
print(f"   → This method will be used in Step 2 for Feature Selection comparison")
print("\n" + "="*80)

## 2. Pick FS / FS X -> FS X

In [ ]:
from sklearn.feature_selection import SelectFromModel

# FEATURE SELECTION
def fs_transform(X_tr, y_tr, X_te, random_state):
    """
    Feature Selection using L1 Regularization + SelectFromModel
    """
    base = LogisticRegression(
        penalty='l1',
        solver='liblinear',
        C=0.1,
        random_state=random_state,
        max_iter=1000
    )
    selector = SelectFromModel(base, threshold='median')
    X_tr_fs = selector.fit_transform(X_tr, y_tr)
    X_te_fs = selector.transform(X_te)
    return X_tr_fs, X_te_fs

# STEP 2: FEATURE SELECTION COMPARISON
print("\n" + "="*80)
print("STEP 2: FEATURE SELECTION COMPARISON")
print("="*80)
print("\nBest Sampling Method from Step 1: Class Weight")
print("Comparison: With Feature Selection vs Without Feature Selection")

# Store results
step2_results = []

# Apply Feature Selection
X_train_fs, X_test_fs = fs_transform(X_train_scaled, y_train, X_test_scaled, randomseed)

print(f"\nFeature Selection: L1 Regularization + SelectFromModel (threshold='median')")
print(f"  Original features: {X_train_scaled.shape[1]}")
print(f"  Selected features: {X_train_fs.shape[1]}")

# 1. WITH FEATURE SELECTION
print("\n" + "#"*80)
print("# COMPARISON 1: CLASS WEIGHT + FEATURE SELECTION")
print("#"*80)

model_fs = LogisticRegression(max_iter=1000, random_state=randomseed, class_weight='balanced')
model_fs.fit(X_train_fs, y_train)

y_pred_fs = model_fs.predict(X_test_fs)
y_proba_fs = model_fs.predict_proba(X_test_fs)[:, 1]

metrics_fs = print_full_results(y_test, y_pred_fs, y_proba_fs, "Class Weight + Feature Selection")
step2_results.append({
    'Method': 'With FS',
    'Num_Features': X_train_fs.shape[1],
    **metrics_fs
})

# 2. WITHOUT FEATURE SELECTION
print("\n" + "#"*80)
print("# COMPARISON 2: CLASS WEIGHT + NO FEATURE SELECTION")
print("#"*80)

model_no_fs = LogisticRegression(max_iter=1000, random_state=randomseed, class_weight='balanced')
model_no_fs.fit(X_train_scaled, y_train)

y_pred_no_fs = model_no_fs.predict(X_test_scaled)
y_proba_no_fs = model_no_fs.predict_proba(X_test_scaled)[:, 1]

metrics_no_fs = print_full_results(y_test, y_pred_no_fs, y_proba_no_fs, "Class Weight + No Feature Selection")
step2_results.append({
    'Method': 'Without FS',
    'Num_Features': X_train_scaled.shape[1],
    **metrics_no_fs
})

# COMPARISON SUMMARY
print("\n" + "="*80)
print("STEP 2: COMPARISON SUMMARY")
print("="*80)

# Create comparison DataFrame
step2_df = pd.DataFrame(step2_results)
step2_df = step2_df.set_index('Method')

column_order = ['Num_Features', 'PR-AUC', 'F2-Score', 'MCC', 'ROC-AUC', 'Accuracy', 'Precision', 'Recall', 'F1-Score']
step2_df = step2_df[column_order]
step2_df = step2_df.round(4)

step2_df_sorted = step2_df.sort_values('PR-AUC', ascending=False)

print("\n📊 Feature Selection Comparison (Sorted by PR-AUC):\n")
print(step2_df_sorted.to_string())

# DETAILED COMPARISON
print("\n" + "-"*80)
print("DETAILED METRIC COMPARISON")
print("-"*80)

fs_row = step2_df.loc['With FS']
no_fs_row = step2_df.loc['Without FS']

print(f"\n{'Metric':<15} {'With FS':>12} {'Without FS':>12} {'Difference':>12} {'Better':>10}")
print("-"*65)

for metric in ['PR-AUC', 'F2-Score', 'MCC', 'ROC-AUC', 'Accuracy', 'Precision', 'Recall', 'F1-Score']:
    fs_val = fs_row[metric]
    no_fs_val = no_fs_row[metric]
    diff = fs_val - no_fs_val
    better = "FS ✓" if diff > 0 else ("No FS ✓" if diff < 0 else "Tie")
    print(f"{metric:<15} {fs_val:>12.4f} {no_fs_val:>12.4f} {diff:>+12.4f} {better:>10}")

# ============================================================================
# BEST CHOICE SELECTION
# ============================================================================
print("\n" + "="*80)
print("🏆 BEST FEATURE SELECTION CHOICE")
print("="*80)

best_fs_choice = step2_df_sorted.index[0]
best_fs_metrics = step2_df_sorted.iloc[0]

print(f"\n  Best Choice: {best_fs_choice}")
print(f"  Number of Features: {int(best_fs_metrics['Num_Features'])}")
print(f"\n  Primary Metrics:")
print(f"    PR-AUC:    {best_fs_metrics['PR-AUC']:.4f}")
print(f"    F2-Score:  {best_fs_metrics['F2-Score']:.4f}")
print(f"    MCC:       {best_fs_metrics['MCC']:.4f}")
print(f"\n  Additional Metrics:")
print(f"    ROC-AUC:   {best_fs_metrics['ROC-AUC']:.4f}")
print(f"    Accuracy:  {best_fs_metrics['Accuracy']:.4f}")
print(f"    Precision: {best_fs_metrics['Precision']:.4f}")
print(f"    Recall:    {best_fs_metrics['Recall']:.4f}")
print(f"    F1-Score:  {best_fs_metrics['F1-Score']:.4f}")

# ============================================================================
# FINAL OUTPUT
# ============================================================================
print("\n" + "="*80)
print("STEP 2 COMPLETE")
print("="*80)
print(f"\n  Step 1 Best: Class Weight")
print(f"  Step 2 Best: {best_fs_choice}")
print(f"\n  → Configuration for Step 3 (Hyperparameter Tuning):")
print(f"     - Sampling: Class Weight")
print(f"     - Feature Selection: {best_fs_choice}")
print("\n" + "="*80)

## 3. Hyperparameter tunning

In [ ]:
!pip install optuna

In [ ]:
"""
================================================================================
STEP 3: HYPERPARAMETER TUNING
================================================================================
- Best Sampling Method from Step 1: Class Weight
- Best Feature Selection from Step 2: Without FS
- Hyperparameter Tuning: GridSearchCV & Optuna
- Scoring: PR-AUC (average_precision)
================================================================================
"""

import numpy as np
import pandas as pd
import optuna
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, cross_val_score, KFold
from sklearn.metrics import classification_report

# ============================================================================
# CONFIGURATION
# ============================================================================
n_trials = 30
C_grid = np.logspace(-3, 3, 7)  # [0.001, 0.01, ..., 1000]

print("\n" + "="*80)
print("STEP 3: HYPERPARAMETER TUNING")
print("="*80)
print("\nConfiguration:")
print(f"  Sampling Method: Class Weight")
print(f"  Feature Selection: Without FS")
print(f"  Scoring: average_precision (PR-AUC)")
print(f"  GridSearchCV C_grid: {C_grid}")
print(f"  Optuna n_trials: {n_trials}")
print(f"  Random Seed: {randomseed}")

# Data: Without FS
X_tr = X_train_scaled
X_te = X_test_scaled
print(f"\nData Info:")
print(f"  Train samples: {X_tr.shape[0]}")
print(f"  Test samples:  {X_te.shape[0]}")
print(f"  Number of Features: {X_tr.shape[1]}")

# CV (Non-Stratified KFold, n_splits=3, shuffle+seed)
cv_kf = KFold(n_splits=3, shuffle=True, random_state=randomseed)

# Store results
step3_results = []

# ============================================================================
# 1. GRIDSEARCHCV
# ============================================================================
print("\n" + "#"*80)
print("# METHOD 1: GRIDSEARCHCV")
print("#"*80)

lr_gs = LogisticRegression(
    max_iter=1000,
    random_state=randomseed,
    class_weight='balanced'
)

param_grid = {'C': C_grid}

gs = GridSearchCV(
    estimator=lr_gs,
    param_grid=param_grid,
    scoring='average_precision',  # PR-AUC
    cv=cv_kf,
    n_jobs=-1,
    verbose=0
)

gs.fit(X_tr, y_train)

y_prob_gs = gs.best_estimator_.predict_proba(X_te)[:, 1]
y_pred_gs = (y_prob_gs > 0.5).astype(int)

metrics_gs = calculate_all_metrics(y_test, y_pred_gs, y_prob_gs)

print(f"\n[GridSearchCV] Class Weight + No FS")
print(f"Best params: {gs.best_params_}")
print(f"Best CV PR-AUC: {gs.best_score_:.4f}")
print(f"Test PR-AUC: {metrics_gs['PR-AUC']:.4f}")
print("\n[Classification Report - GridSearchCV]")
print(classification_report(y_test, y_pred_gs, digits=4))

step3_results.append({
    'Method': 'GridSearchCV',
    'Best_C': gs.best_params_['C'],
    'CV_PR-AUC': gs.best_score_,
    **metrics_gs
})

# Full result (PR-AUC, F2, MCC, etc. + Confusion Matrix + Report)
print_full_results(y_test, y_pred_gs, y_prob_gs, "GridSearchCV Best Model")

# ============================================================================
# 2. OPTUNA
# ============================================================================
print("\n" + "#"*80)
print("# METHOD 2: OPTUNA")
print("#"*80)

def objective(trial):
    C = trial.suggest_float('C', 1e-3, 1e3, log=True)
    model = LogisticRegression(
        C=C,
        class_weight='balanced',
        max_iter=1000,
        random_state=randomseed
    )
    scores = cross_val_score(
        model,
        X_tr,
        y_train,
        cv=cv_kf,
        scoring='average_precision',  # PR-AUC
        n_jobs=-1
    )
    return scores.mean()

optuna.logging.set_verbosity(optuna.logging.WARNING)

# TPESampler X
study = optuna.create_study(
    direction='maximize'
)
study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

# Train best model on full train set
best_model_optuna = LogisticRegression(
    C=study.best_params['C'],
    class_weight='balanced',
    max_iter=1000,
    random_state=randomseed
)
best_model_optuna.fit(X_tr, y_train)

y_prob_optuna = best_model_optuna.predict_proba(X_te)[:, 1]
y_pred_optuna = (y_prob_optuna > 0.5).astype(int)

metrics_optuna = calculate_all_metrics(y_test, y_pred_optuna, y_prob_optuna)

print(f"\n[Optuna] Class Weight + No FS")
print(f"Best params: {study.best_params}")
print(f"Best CV PR-AUC: {study.best_value:.4f}")
print(f"Test PR-AUC: {metrics_optuna['PR-AUC']:.4f}")
print("\n[Classification Report - Optuna]")
print(classification_report(y_test, y_pred_optuna, digits=4))

step3_results.append({
    'Method': 'Optuna',
    'Best_C': study.best_params['C'],
    'CV_PR-AUC': study.best_value,
    **metrics_optuna
})

# Full result (PR-AUC, F2, MCC, etc. + Confusion Matrix + Report)
print_full_results(y_test, y_pred_optuna, y_prob_optuna, "Optuna Best Model")

# ============================================================================
# 3. COMPARISON SUMMARY
# ============================================================================
print("\n" + "="*80)
print("STEP 3: COMPARISON SUMMARY")
print("="*80)

step3_df = pd.DataFrame(step3_results).set_index('Method')

column_order = [
    'Best_C', 'CV_PR-AUC', 'PR-AUC', 'F2-Score', 'MCC',
    'ROC-AUC', 'Accuracy', 'Precision', 'Recall', 'F1-Score'
]
step3_df = step3_df[column_order].round(4)

step3_df_sorted = step3_df.sort_values('PR-AUC', ascending=False)

print("\n📊 Hyperparameter Tuning Comparison (Sorted by Test PR-AUC):\n")
print(step3_df_sorted.to_string())

# ============================================================================
# 4. DETAILED COMPARISON
# ============================================================================
print("\n" + "-"*80)
print("DETAILED METRIC COMPARISON")
print("-"*80)

print(f"\n{'Method':<15} {'Best C':>12} {'CV PR-AUC':>12} {'Test PR-AUC':>12} {'F2-Score':>12} {'MCC':>12}")
print("-"*80)

for method in step3_df_sorted.index:
    row = step3_df_sorted.loc[method]
    print(f"{method:<15} {row['Best_C']:>12.4f} {row['CV_PR-AUC']:>12.4f} "
          f"{row['PR-AUC']:>12.4f} {row['F2-Score']:>12.4f} {row['MCC']:>12.4f}")

# ============================================================================
# 5. BEST MODEL SELECTION
# ============================================================================
print("\n" + "="*80)
print("🏆 BEST HYPERPARAMETER TUNING METHOD")
print("="*80)

best_method = step3_df_sorted.index[0]
best_metrics = step3_df_sorted.iloc[0]

print(f"\n  Best Method: {best_method}")
print(f"  Best C: {best_metrics['Best_C']:.6f}")
print(f"\n  Cross-Validation:")
print(f"    CV PR-AUC: {best_metrics['CV_PR-AUC']:.4f}")
print(f"\n  Test Set Primary Metrics:")
print(f"    PR-AUC:    {best_metrics['PR-AUC']:.4f}")
print(f"    F2-Score:  {best_metrics['F2-Score']:.4f}")
print(f"    MCC:       {best_metrics['MCC']:.4f}")
print(f"\n  Test Set Additional Metrics:")
print(f"    ROC-AUC:   {best_metrics['ROC-AUC']:.4f}")
print(f"    Accuracy:  {best_metrics['Accuracy']:.4f}")
print(f"    Precision: {best_metrics['Precision']:.4f}")
print(f"    Recall:    {best_metrics['Recall']:.4f}")
print(f"    F1-Score:  {best_metrics['F1-Score']:.4f}")

# ============================================================================
# 6. FINAL OUTPUT
# ============================================================================
print("\n" + "="*80)
print("STEP 3 COMPLETE - FINAL MODEL CONFIGURATION")
print("="*80)
print(f"\n  Step 1: Sampling Method = Class Weight")
print(f"  Step 2: Feature Selection = Without FS")
print(f"  Step 3: Best Tuning = {best_method}")
print(f"\n  📌 FINAL MODEL PARAMETERS:")
print(f"     - C: {best_metrics['Best_C']:.6f}")
print(f"     - class_weight: 'balanced'")
print(f"     - max_iter: 1000")
print(f"     - random_state: {randomseed}")
print(f"\n  📌 FINAL MODEL PERFORMANCE (Test Set):")
print(f"     - PR-AUC:   {best_metrics['PR-AUC']:.4f}")
print(f"     - F2-Score: {best_metrics['F2-Score']:.4f}")
print(f"     - MCC:      {best_metrics['MCC']:.4f}")
print("\n" + "="*80)